# 📊 Notebook 01 — Data Analysis & Valence-Arousal Mapping

## Obiettivo
Questo notebook carica i **due dataset di partenza** (Student Life e 14-Day Depression Log),
ne esplora la struttura, calcola il **punteggio PHQ-9** e mappa tutte le metriche 
emotive disponibili sullo spazio **Valenza-Arousal** del Modello Circomplesso di Russell.

Le statistiche estratte (matrici di transizione, distribuzioni di mood per fascia PHQ-9,
tassi di dropout e volatilità) verranno salvate come artefatti JSON da utilizzare nel
Notebook 02 per la generazione sintetica.

### Dataset di partenza
| Nome File | Sorgente | Contenuto |
|---|---|---|
| `studentlife_phq9_scores.csv` | Student Life (Dartmouth) | Questionario PHQ-9 pre/post per ~48 studenti |
| `studentlife_mood_detailed/*.json` | Student Life (Dartmouth) | Check-in umore (happy 1-4, sad 1-4) con timestamp |
| `studentlife_mood_prediction/*.json` | Student Life (Dartmouth) | Previsione umore del giorno dopo (1=happy,2=stressed,3=tired) |
| `studentlife_mood_current/*.json` | Student Life (Dartmouth) | Stato attuale (1=happy,2=stressed,3=tired) |
| `14day_ema_depression_log.csv` | 14-day AA study | EMA giornaliero, PHQ-9 per paziente, happiness.score |

## 1. Import e Configurazione

In [1]:
import os
import json
import glob
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

RAW_DIR = os.path.join('..', 'data', 'raw')
PROCESSED_DIR = os.path.join('..', 'data', 'processed')
FIGURES_DIR = os.path.join('..', '..', '..', 'docs', 'documentazione', 'latex', 'figures')
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print("✅ Librerie caricate correttamente")

✅ Librerie caricate correttamente


---
## 2. Il Modello Circomplesso di Russell: Mappatura Valence-Arousal

Per unificare le diverse scale emotive dei dataset e renderle compatibili con i 
**10 stati d'animo × 10 intensità** di SINTON-IA, utilizziamo il **Modello Circomplesso
dell'Affetto** di James A. Russell (1980).

Ogni emozione viene proiettata su due assi:
- **Valence** (Valenza): da -1 (molto negativo) a +1 (molto positivo)
- **Arousal** (Attivazione): da -1 (passivo/letargico) a +1 (attivo/energetico)

### Mappatura degli stati SINTON-IA
I 10 stati d'animo dell'app vengono definiti come coordinate fisse nello spazio V-A,
basate sulla letteratura psicologica (Russell, 1980; Posner et al., 2005).

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# MAPPATURA SINTON-IA → VALENCE / AROUSAL  (Modello Circomplesso)
# ═══════════════════════════════════════════════════════════════════
# Coordinate basate su:
#   - Russell, J.A. (1980). A circumplex model of affect.
#   - Posner, J. et al. (2005). The circumplex model of affect.
#   - Barrett, L.F. & Russell, J.A. (1999). The structure of current affect.

SINTONIA_VALENCE_AROUSAL = {
    # Stato d'animo     : (Valence, Arousal)
    'Felice'            : ( 0.85,   0.60),
    'Sereno'            : ( 0.70,  -0.20),
    'Energico'          : ( 0.50,   0.90),
    'Neutro'            : ( 0.00,   0.00),
    'Stanco'            : (-0.20,  -0.75),
    'Triste'            : (-0.80,  -0.40),
    'Ansioso'           : (-0.55,   0.70),
    'Arrabbiato'        : (-0.70,   0.80),
    'Spaventato'        : (-0.65,   0.75),
    'Confuso'           : (-0.30,   0.20),
}

print("╔══════════════════════════════════════════════════════════╗")
print("║   MAPPATURA STATI D'ANIMO SINTON-IA → VALENCE/AROUSAL  ║")
print("╠══════════════════╤══════════╤══════════╤════════════════╣")
print("║ Stato d'animo    │ Valence  │ Arousal  │ Quadrante      ║")
print("╟──────────────────┼──────────┼──────────┼────────────────╢")
for name, (v, a) in SINTONIA_VALENCE_AROUSAL.items():
    if v >= 0 and a >= 0:
        quad = "Alto-Positivo"
    elif v >= 0 and a < 0:
        quad = "Basso-Positivo"
    elif v < 0 and a >= 0:
        quad = "Alto-Negativo"
    else:
        quad = "Basso-Negativo"
    print(f"║ {name:<16} │ {v:>+6.2f}   │ {a:>+6.2f}   │ {quad:<14} ║")
print("╚══════════════════╧══════════╧══════════╧════════════════╝")

# Visualizzazione grafica nello spazio V-A
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
ax.set_xlim(-1.2, 1.2)
ax.set_ylim(-1.2, 1.2)
ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.5, linestyle='--')

colors = {
    'Felice': '#FFD700', 'Sereno': '#87CEEB', 'Energico': '#FF6347',
    'Neutro': '#C0C0C0', 'Stanco': '#8B8B83', 'Triste': '#4169E1',
    'Ansioso': '#FF8C00', 'Arrabbiato': '#DC143C', 'Spaventato': '#9400D3',
    'Confuso': '#DDA0DD'
}

for name, (v, a) in SINTONIA_VALENCE_AROUSAL.items():
    ax.scatter(v, a, s=350, c=colors[name], edgecolors='black', linewidths=1.5, zorder=5)
    ax.annotate(name, (v, a), textcoords="offset points", xytext=(0, 15),
                ha='center', fontsize=11, fontweight='bold')

circle = plt.Circle((0, 0), 1, fill=False, linestyle=':', color='gray', linewidth=1)
ax.add_patch(circle)
ax.set_xlabel('← Valenza Negativa — Valenza Positiva →', fontsize=13)
ax.set_ylabel('← Bassa Attivazione — Alta Attivazione →', fontsize=13)
ax.set_title('Spazio Circomplesso di Russell: Mappatura SINTON-IA', fontsize=15, fontweight='bold')
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'dp_valence_arousal_map.png'), dpi=150, bbox_inches='tight')
plt.close()
print("✅ Mappa Valence-Arousal salvata in docs/documentazione/latex/figures/dp_valence_arousal_map.png")

╔══════════════════════════════════════════════════════════╗
║   MAPPATURA STATI D'ANIMO SINTON-IA → VALENCE/AROUSAL  ║
╠══════════════════╤══════════╤══════════╤════════════════╣
║ Stato d'animo    │ Valence  │ Arousal  │ Quadrante      ║
╟──────────────────┼──────────┼──────────┼────────────────╢
║ Felice           │  +0.85   │  +0.60   │ Alto-Positivo  ║
║ Sereno           │  +0.70   │  -0.20   │ Basso-Positivo ║
║ Energico         │  +0.50   │  +0.90   │ Alto-Positivo  ║
║ Neutro           │  +0.00   │  +0.00   │ Alto-Positivo  ║
║ Stanco           │  -0.20   │  -0.75   │ Basso-Negativo ║
║ Triste           │  -0.80   │  -0.40   │ Basso-Negativo ║
║ Ansioso          │  -0.55   │  +0.70   │ Alto-Negativo  ║
║ Arrabbiato       │  -0.70   │  +0.80   │ Alto-Negativo  ║
║ Spaventato       │  -0.65   │  +0.75   │ Alto-Negativo  ║
║ Confuso          │  -0.30   │  +0.20   │ Alto-Negativo  ║
╚══════════════════╧══════════╧══════════╧════════════════╝
✅ Mappa Valence-Arousal salvata in docs/

---
## 3. Caricamento Dataset 1: Student Life (PHQ-9 + Mood JSON)

### 3.1 PHQ-9 — Questionario sulla Depressione
Il Patient Health Questionnaire (PHQ-9) è lo standard clinico per lo screening 
della depressione. Consiste in 9 domande, ciascuna valutata su una scala a 4 punti:
- **0** = "Not at all" (Per niente)
- **1** = "Several days" (Alcuni giorni)
- **2** = "More than half the days" (Più della metà dei giorni)
- **3** = "Nearly every day" (Quasi ogni giorno)

Il punteggio totale va da 0 a 27.

| Range PHQ-9 | Severità |
|---|---|
| 0–4 | Minima |
| 5–9 | Lieve |
| 10–14 | Moderata |
| 15–19 | Moderatamente severa |
| 20–27 | Severa |

In [3]:
# ═══════════════════════════════════════════════════════════════
# 3.1  CARICAMENTO E SCORING PHQ-9 (Student Life)
# ═══════════════════════════════════════════════════════════════
phq9_raw = pd.read_csv(os.path.join(RAW_DIR, 'studentlife_phq9_scores.csv'))
print(f"PHQ-9 dataset: {phq9_raw.shape[0]} righe × {phq9_raw.shape[1]} colonne")
print(f"Colonne: {list(phq9_raw.columns)}")
print(f"\nStudenti unici: {phq9_raw['uid'].nunique()}")
print(f"Tipologie: {phq9_raw['type'].unique()}")
print(f"\nPrime 5 righe:")
phq9_raw.head()

PHQ-9 dataset: 84 righe × 12 colonne
Colonne: ['uid', 'type', 'Little interest or pleasure in doing things', 'Feeling down, depressed, hopeless.', 'Trouble falling or staying asleep, or sleeping too much.', 'Feeling tired or having little energy', 'Poor appetite or overeating', 'Feeling bad about yourself or that you are a failure or have let yourself or your family down', 'Trouble concentrating on things, such as reading the newspaper or watching television', 'Moving or speaking so slowly that other people could have noticed. Or the opposite being so figety or restless that you have been moving around a lot more than usual', 'Thoughts that you would be better off dead, or of hurting yourself', 'Response']

Studenti unici: 46
Tipologie: <ArrowStringArray>
['pre', 'post']
Length: 2, dtype: str

Prime 5 righe:


,uid,type,Little interest or pleasure in doing things,"Feeling down, depressed, hopeless.","Trouble falling or staying asleep, or sleeping too much.",Feeling tired or having little energy,Poor appetite or overeating,Feeling bad about yourself or that you are a failure or have let yourself or your family down,"Trouble concentrating on things, such as reading the newspaper or watching television",Moving or speaking so slowly that other people could have noticed. Or the opposite being so figety or restless that you have been moving around a lot more than usual,"Thoughts that you would be better off dead, or of hurting yourself",Response
0,u00,pre,Not at all,Several days,Not at all,Several days,Not at all,Not at all,Not at all,Not at all,Not at all,Not difficult at all
1,u01,pre,Several days,Several days,Several days,Several days,Not at all,Several days,Not at all,Not at all,Not at all,Very difficult
2,u02,pre,More than half the days,Several days,More than half the days,More than half the days,More than half the days,Several days,Several days,More than half the days,Not at all,Somewhat difficult
3,u03,pre,Not at all,Several days,Not at all,Not at all,Not at all,Not at all,Not at all,Several days,Not at all,Somewhat difficult
4,u04,pre,Several days,Several days,Not at all,Several days,Several days,Several days,Several days,Not at all,Not at all,Somewhat difficult


In [4]:
# Conversione risposte testuali → punteggi numerici PHQ-9
phq9_scoring = {
    'Not at all': 0,
    'Several days': 1,
    'More than half the days': 2,
    'Nearly every day': 3
}

phq9_questions = [
    'Little interest or pleasure in doing things',
    'Feeling down, depressed, hopeless.',
    'Trouble falling or staying asleep, or sleeping too much.',
    'Feeling tired or having little energy',
    'Poor appetite or overeating',
    'Feeling bad about yourself or that you are a failure or have let yourself or your family down',
    'Trouble concentrating on things, such as reading the newspaper or watching television',
    'Moving or speaking so slowly that other people could have noticed. Or the opposite being so figety or restless that you have been moving around a lot more than usual',
    'Thoughts that you would be better off dead, or of hurting yourself'
]

# Scoring
phq9_scored = phq9_raw.copy()
for col in phq9_questions:
    phq9_scored[col] = phq9_scored[col].map(phq9_scoring)

# Calcolo punteggio totale
phq9_scored['phq9_total'] = phq9_scored[phq9_questions].sum(axis=1)

# Classificazione severità
def classify_severity(score):
    if pd.isna(score):
        return 'Unknown'
    score = int(score)
    if score <= 4:
        return 'Minima (0-4)'
    elif score <= 9:
        return 'Lieve (5-9)'
    elif score <= 14:
        return 'Moderata (10-14)'
    elif score <= 19:
        return 'Mod. Severa (15-19)'
    else:
        return 'Severa (20-27)'

phq9_scored['severity'] = phq9_scored['phq9_total'].apply(classify_severity)

print("═" * 60)
print("DISTRIBUZIONE PHQ-9 PER TIPOLOGIA (PRE/POST)")
print("═" * 60)
for t in ['pre', 'post']:
    subset = phq9_scored[phq9_scored['type'] == t]
    print(f"\n▶ {t.upper()} ({len(subset)} studenti)")
    print(f"  Media PHQ-9: {subset['phq9_total'].mean():.2f}")
    print(f"  Mediana:     {subset['phq9_total'].median():.1f}")
    print(f"  Std Dev:     {subset['phq9_total'].std():.2f}")
    print(f"  Range:       [{subset['phq9_total'].min():.0f} – {subset['phq9_total'].max():.0f}]")
    print(f"  Severità:")
    for sev, count in subset['severity'].value_counts().items():
        print(f"    {sev}: {count}")

════════════════════════════════════════════════════════════
DISTRIBUZIONE PHQ-9 PER TIPOLOGIA (PRE/POST)
════════════════════════════════════════════════════════════

▶ PRE (46 studenti)
  Media PHQ-9: 5.52
  Mediana:     5.0
  Std Dev:     4.61
  Range:       [0 – 23]
  Severità:
    Minima (0-4): 21
    Lieve (5-9): 17
    Moderata (10-14): 6
    Mod. Severa (15-19): 1
    Severa (20-27): 1

▶ POST (38 studenti)
  Media PHQ-9: 6.26
  Mediana:     4.5
  Std Dev:     5.84
  Range:       [0 – 25]
  Severità:
    Minima (0-4): 19
    Lieve (5-9): 12
    Moderata (10-14): 3
    Mod. Severa (15-19): 2
    Severa (20-27): 2


In [5]:
# Visualizzazione distribuzione PHQ-9 pre vs post
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, t in enumerate(['pre', 'post']):
    subset = phq9_scored[phq9_scored['type'] == t]
    ax = axes[idx]
    
    color_map = {
        'Minima (0-4)': '#2ecc71', 'Lieve (5-9)': '#f39c12',
        'Moderata (10-14)': '#e67e22', 'Mod. Severa (15-19)': '#e74c3c',
        'Severa (20-27)': '#c0392b'
    }
    
    ax.hist(subset['phq9_total'].dropna(), bins=range(0, 28), 
            color='#3498db' if t == 'pre' else '#e74c3c', alpha=0.7, edgecolor='black')
    ax.set_xlabel('Punteggio PHQ-9')
    ax.set_ylabel('Numero Studenti')
    ax.set_title(f'Distribuzione PHQ-9 — {t.upper()}', fontweight='bold')
    ax.axvline(subset['phq9_total'].mean(), color='red', linestyle='--', label=f"Media: {subset['phq9_total'].mean():.1f}")
    ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'dp_phq9_distribution.png'), dpi=150, bbox_inches='tight')
plt.close()

### 3.2 Mood JSON — Check-in Emotivi degli Studenti

Ogni file `Mood_uXX.json` contiene registrazioni con:
- `happy` (1-4): intensità felicità (1=un po', 4=estremamente)
- `sad` (1-4): intensità tristezza
- `happyornot` (1=Sì, 2=No): flag binario felicità
- `sadornot` (1=Sì, 2=No): flag binario tristezza
- `resp_time`: timestamp UNIX
- `location`: coordinate GPS

**Mood 1** contiene `tomorrow` (1=happy, 2=stressed, 3=tired): previsione del giorno dopo.
**Mood 2** contiene `how` (1=happy, 2=stressed, 3=tired): stato attuale.

In [6]:
# ═══════════════════════════════════════════════════════════════
# 3.2  CARICAMENTO MOOD JSON (Student Life)
# ═══════════════════════════════════════════════════════════════

def load_mood_folder(folder_path, mood_type='mood'):
    """Carica tutti i file JSON di una cartella Mood e li concatena."""
    all_records = []
    pattern = os.path.join(folder_path, '*.json')
    
    for filepath in sorted(glob.glob(pattern)):
        filename = os.path.basename(filepath)
        # Estrai user_id dal nome file (es: "Mood_u00.json" -> "u00")
        uid = filename.split('_')[-1].replace('.json', '')
        
        try:
            with open(filepath, 'r') as f:
                data = json.load(f)
            
            if isinstance(data, list):
                for record in data:
                    record['uid'] = uid
                    record['mood_type'] = mood_type
                    all_records.append(record)
        except (json.JSONDecodeError, Exception):
            pass
    
    return pd.DataFrame(all_records)

# Caricamento dei 3 tipi di Mood
mood_df = load_mood_folder(os.path.join(RAW_DIR, 'studentlife_mood_detailed'), 'mood_detailed')
mood1_df = load_mood_folder(os.path.join(RAW_DIR, 'studentlife_mood_prediction'), 'mood_prediction')
mood2_df = load_mood_folder(os.path.join(RAW_DIR, 'studentlife_mood_current'), 'mood_current')

print("═" * 60)
print("RIEPILOGO MOOD DATA (Student Life)")
print("═" * 60)
print(f"\n▶ Mood (dettagliato - happy/sad scale):  {len(mood_df):>6} record, {mood_df['uid'].nunique():>3} studenti")
print(f"▶ Mood 1 (previsione domani):             {len(mood1_df):>6} record, {mood1_df['uid'].nunique():>3} studenti")
print(f"▶ Mood 2 (stato attuale):                 {len(mood2_df):>6} record, {mood2_df['uid'].nunique():>3} studenti")

════════════════════════════════════════════════════════════
RIEPILOGO MOOD DATA (Student Life)
════════════════════════════════════════════════════════════

▶ Mood (dettagliato - happy/sad scale):     277 record,  38 studenti
▶ Mood 1 (previsione domani):                270 record,  40 studenti
▶ Mood 2 (stato attuale):                    426 record,  39 studenti


In [7]:
 # Analisi del Mood dettagliato
if len(mood_df) > 0:
    # Converti tipi
    for col in ['happy', 'sad']:
        mood_df[col] = pd.to_numeric(mood_df[col], errors='coerce')
    
    # Converti timestamp
    mood_df['datetime'] = pd.to_datetime(mood_df['resp_time'], unit='s', errors='coerce')
    
    print("\n═" * 60)
    print("STATISTICHE MOOD DETTAGLIATO")
    print("═" * 60)
    print(f"\n  Happy scale (1-4):")
    print(f"    Media:   {mood_df['happy'].mean():.2f}")
    print(f"    Mediana: {mood_df['happy'].median():.1f}")
    print(f"    Distribuzione:")
    for val, count in mood_df['happy'].value_counts().sort_index().items():
        print(f"      {val}: {count} ({count/len(mood_df)*100:.1f}%)")
    
    print(f"\n  Sad scale (1-4):")
    print(f"    Media:   {mood_df['sad'].mean():.2f}")
    print(f"    Mediana: {mood_df['sad'].median():.1f}")
    print(f"    Distribuzione:")
    for val, count in mood_df['sad'].value_counts().sort_index().items():
        print(f"      {val}: {count} ({count/len(mood_df)*100:.1f}%)")


═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
STATISTICHE MOOD DETTAGLIATO
════════════════════════════════════════════════════════════

  Happy scale (1-4):
    Media:   1.87
    Mediana: 2.0
    Distribuzione:
      1.0: 124 (44.8%)
      2.0: 80 (28.9%)
      3.0: 57 (20.6%)
      4.0: 15 (5.4%)

  Sad scale (1-4):
    Media:   1.37
    Mediana: 1.0
    Distribuzione:
      1.0: 218 (78.7%)
      2.0: 26 (9.4%)
      3.0: 19 (6.9%)
      4.0: 13 (4.7%)


### 3.3 Mappatura Student Life → Valence-Arousal

Trasformiamo i punteggi `happy` e `sad` del dataset originale nello spazio V-A:
- **Valence** = (happy_norm - sad_norm), dove ogni score è normalizzato [0,1]
- **Arousal** = derivato dall'intensità complessiva dell'emozione

In [8]:
# ═══════════════════════════════════════════════════════════════
# 3.3  PROIEZIONE STUDENT LIFE → VALENCE-AROUSAL
# ═══════════════════════════════════════════════════════════════

def student_life_to_VA(happy, sad):
    """
    Proietta i punteggi happy (1-4) e sad (1-4) del Student Life
    nello spazio Valence-Arousal.
    
    Logica basata su Barrett & Russell (1999):
    - Valence: differenza normalizzata tra felicità e tristezza
    - Arousal: intensità emotiva complessiva (più è intenso, più è attivo)
    """
    # Normalizzazione [0, 1]
    h_norm = (happy - 1) / 3  # 1→0, 4→1
    s_norm = (sad - 1) / 3    # 1→0, 4→1
    
    # Valence: asse positivo-negativo
    valence = h_norm - s_norm  # Range: [-1, 1]
    
    # Arousal: intensità emotiva (alta felicità O alta tristezza = alta attivazione)
    arousal = max(h_norm, s_norm) * 2 - 1  # Più l'emozione è forte, maggiore l'arousal
    # Correggiamo: tristezza intensa abbassa leggermente l'arousal
    if s_norm > h_norm:
        arousal = arousal * 0.7  # La tristezza tende alla passività
    
    return valence, arousal

if len(mood_df) > 0:
    mood_df['valence'], mood_df['arousal'] = zip(*mood_df.apply(
        lambda r: student_life_to_VA(r['happy'], r['sad']) 
        if pd.notna(r['happy']) and pd.notna(r['sad']) else (np.nan, np.nan), axis=1))
    
    # Unisci con PHQ-9 per analisi per fasce di severità
    phq9_pre = phq9_scored[phq9_scored['type'] == 'pre'][['uid', 'phq9_total', 'severity']]
    mood_with_phq = mood_df.merge(phq9_pre, on='uid', how='left')
    
    print("═" * 60)
    print("ANALISI VALENCE-AROUSAL PER SEVERITÀ PHQ-9")
    print("═" * 60)
    for sev in ['Minima (0-4)', 'Lieve (5-9)', 'Moderata (10-14)', 'Mod. Severa (15-19)', 'Severa (20-27)']:
        subset = mood_with_phq[mood_with_phq['severity'] == sev]
        if len(subset) > 0:
            print(f"\n▶ {sev} ({len(subset)} check-in)")
            print(f"  Valence media: {subset['valence'].mean():+.3f}")
            print(f"  Arousal medio: {subset['arousal'].mean():+.3f}")
            print(f"  Valence std:   {subset['valence'].std():.3f}")

════════════════════════════════════════════════════════════
ANALISI VALENCE-AROUSAL PER SEVERITÀ PHQ-9
════════════════════════════════════════════════════════════

▶ Minima (0-4) (77 check-in)
  Valence media: +0.211
  Arousal medio: -0.450
  Valence std:   0.312

▶ Lieve (5-9) (176 check-in)
  Valence media: +0.225
  Arousal medio: -0.179
  Valence std:   0.457

▶ Moderata (10-14) (16 check-in)
  Valence media: -0.417
  Arousal medio: +0.102
  Valence std:   0.551

▶ Mod. Severa (15-19) (1 check-in)
  Valence media: -0.333
  Arousal medio: -0.233
  Valence std:   nan

▶ Severa (20-27) (7 check-in)
  Valence media: -0.476
  Arousal medio: +0.367
  Valence std:   0.634


---
## 4. Caricamento Dataset 2: 14-Day Depression Log

Questo dataset contiene dati EMA (Ecological Momentary Assessment) raccolti su 14 giorni
con punteggi PHQ-9 per paziente, risposte sintomatologiche (q1-q14, q16, q46, q47)
e un `happiness.score` (0-4).

In [9]:
# ═══════════════════════════════════════════════════════════════
# 4.  CARICAMENTO 14-DAY CSV
# ═══════════════════════════════════════════════════════════════
csv14_path = os.path.join(RAW_DIR, '14day_ema_depression_log.csv')
df14 = pd.read_csv(csv14_path)

print(f"14-Day dataset: {df14.shape[0]} righe × {df14.shape[1]} colonne")
print(f"Pazienti unici: {df14['user_id'].nunique()}")
print(f"\nColonne: {list(df14.columns)}")

14-Day dataset: 16150 righe × 36 colonne
Pazienti unici: 185

Colonne: ['Unnamed: 0', 'user_id', 'phq1', 'phq2', 'phq3', 'phq4', 'phq5', 'phq6', 'phq7', 'phq8', 'phq9', 'age', 'sex', 'q1', 'q2', 'q3', 'q4', 'q5', 'q6', 'q7', 'q8', 'q9', 'q10', 'q11', 'q12', 'q13', 'q14', 'q16', 'q46', 'q47', 'happiness.score', 'time', 'period.name', 'start.time', 'phq.day', 'id']


In [10]:
# Calcolo PHQ-9 totale per il 14-day dataset
phq_cols_14d = ['phq1','phq2','phq3','phq4','phq5','phq6','phq7','phq8','phq9']
df14['phq9_total'] = df14[phq_cols_14d].sum(axis=1)
df14['severity'] = df14['phq9_total'].apply(classify_severity)

# Statistiche pazienti (un record per user_id con il primo phq9_total)
patients_14d = df14.groupby('user_id').agg(
    phq9_total=('phq9_total', 'first'),
    severity=('severity', 'first'),
    n_records=('user_id', 'count'),
    happiness_mean=('happiness.score', 'mean'),
    happiness_std=('happiness.score', 'std')
).reset_index()

print("\n═" * 60)
print("DISTRIBUZIONE SEVERITÀ — 14-DAY DATASET")
print("═" * 60)
for sev, count in patients_14d['severity'].value_counts().sort_index().items():
    print(f"  {sev}: {count} pazienti ({count/len(patients_14d)*100:.1f}%)")

print(f"\nMedia PHQ-9 totale:  {patients_14d['phq9_total'].mean():.2f}")
print(f"Mediana:             {patients_14d['phq9_total'].median():.1f}")
print(f"Record medi/pazient: {patients_14d['n_records'].mean():.1f}")


═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
DISTRIBUZIONE SEVERITÀ — 14-DAY DATASET
════════════════════════════════════════════════════════════
  Lieve (5-9): 34 pazienti (18.4%)
  Minima (0-4): 12 pazienti (6.5%)
  Mod. Severa (15-19): 44 pazienti (23.8%)
  Moderata (10-14): 41 pazienti (22.2%)
  Severa (20-27): 54 pazienti (29.2%)

Media PHQ-9 totale:  14.85
Mediana:             16.0
Record medi/pazient: 87.3


In [11]:
# Mappatura happiness.score (0-4) del 14-day → Valence
# Nel 14-day dataset, happiness.score: 0=molto infelice → 4=molto felice

def happiness_score_to_VA(score):
    """
    Converte happiness.score (0-4) → (Valence, Arousal).
    Il 14-day dataset ha solo un asse di felicità, quindi:
    - Valence = mappatura lineare [0,4] → [-1,1]
    - Arousal = stima basata sulla distanza dal neutro
    """
    valence = (score / 4) * 2 - 1   # 0→-1, 2→0, 4→+1
    arousal = abs(valence) * 0.5 - 0.25  # Estremi emotivi = più attivazione
    return valence, arousal

df14['valence'], df14['arousal'] = zip(*df14['happiness.score'].apply(
    lambda s: happiness_score_to_VA(s) if pd.notna(s) else (np.nan, np.nan)))

print("\n═" * 60)
print("ANALISI VALENCE PER SEVERITÀ — 14-DAY")
print("═" * 60)
for sev in ['Minima (0-4)', 'Lieve (5-9)', 'Moderata (10-14)', 'Mod. Severa (15-19)', 'Severa (20-27)']:
    subset = df14[df14['severity'] == sev]
    if len(subset) > 0:
        print(f"\n▶ {sev}")
        print(f"  Valence media:     {subset['valence'].mean():+.3f}")
        print(f"  Happiness media:   {subset['happiness.score'].mean():.2f}")
        print(f"  Volatilità (std):  {subset['happiness.score'].std():.3f}")


═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
ANALISI VALENCE PER SEVERITÀ — 14-DAY
════════════════════════════════════════════════════════════

▶ Minima (0-4)
  Valence media:     +0.086
  Happiness media:   2.17
  Volatilità (std):  0.846

▶ Lieve (5-9)
  Valence media:     +0.202
  Happiness media:   2.40
  Volatilità (std):  0.727

▶ Moderata (10-14)
  Valence media:     +0.088
  Happiness media:   2.18
  Volatilità (std):  0.939

▶ Mod. Severa (15-19)
  Valence media:     -0.214
  Happiness media:   1.57
  Volatilità (std):  0.915

▶ Severa (20-27)
  Valence media:     -0.296
  Happiness media:   1.41
  Volatilità (std):  0.975


---
## 5. Estrazione Pattern Statistici per la Generazione Sintetica

Estraiamo le **statistiche chiave** che guideranno l'algoritmo di generazione 
nel Notebook 02:
1. **Distribuzione della Valence** per ciascuna fascia di gravità PHQ-9
2. **Volatilità emotiva** (deviazione standard) per fascia
3. **Matrice di Transizione** degli stati emotivi (catena di Markov)
4. **Tasso di Missing Data** (dropout rate) per fascia di gravità
5. **Durata media delle streak negative**

In [12]:
# ═══════════════════════════════════════════════════════════════
# 5.1  DISTRIBUZIONE VALENCE PER FASCIA PHQ-9
# ═══════════════════════════════════════════════════════════════
stats_by_severity = {}

severities = ['Minima (0-4)', 'Lieve (5-9)', 'Moderata (10-14)', 'Mod. Severa (15-19)', 'Severa (20-27)']

for sev in severities:
    subset = df14[df14['severity'] == sev]
    if len(subset) > 0:
        stats_by_severity[sev] = {
            'n_patients': int(subset['user_id'].nunique()),
            'n_records': int(len(subset)),
            'valence_mean': float(subset['valence'].mean()),
            'valence_std': float(subset['valence'].std()),
            'valence_median': float(subset['valence'].median()),
            'arousal_mean': float(subset['arousal'].mean()),
            'happiness_mean': float(subset['happiness.score'].mean()),
            'happiness_std': float(subset['happiness.score'].std()),
        }

print("═" * 70)
print("RIEPILOGO STATISTICHE ESTRATTE PER FASCIA")
print("═" * 70)
print(f"{'Severità':<22} {'N.Paz':>6} {'V.Mean':>8} {'V.Std':>7} {'H.Mean':>7} {'H.Std':>7}")
print("─" * 70)
for sev, s in stats_by_severity.items():
    print(f"{sev:<22} {s['n_patients']:>6} {s['valence_mean']:>+8.3f} {s['valence_std']:>7.3f} "
          f"{s['happiness_mean']:>7.2f} {s['happiness_std']:>7.3f}")

══════════════════════════════════════════════════════════════════════
RIEPILOGO STATISTICHE ESTRATTE PER FASCIA
══════════════════════════════════════════════════════════════════════
Severità                N.Paz   V.Mean   V.Std  H.Mean   H.Std
──────────────────────────────────────────────────────────────────────
Minima (0-4)               12   +0.086   0.423    2.17   0.846
Lieve (5-9)                34   +0.202   0.363    2.40   0.727
Moderata (10-14)           41   +0.088   0.470    2.18   0.939
Mod. Severa (15-19)        44   -0.214   0.458    1.57   0.915
Severa (20-27)             54   -0.296   0.488    1.41   0.975


In [13]:
# ═══════════════════════════════════════════════════════════════
# 5.2  MATRICE DI TRANSIZIONE EMOTIVA (Catena di Markov 1° ordine)
# ═══════════════════════════════════════════════════════════════
# Discretizziamo la valence in 5 livelli per calcolare le transizioni 

def discretize_valence(v):
    """Discretizza la valence in 5 categorie."""
    if v <= -0.6:
        return 'Molto_Negativo'
    elif v <= -0.2:
        return 'Negativo'
    elif v <= 0.2:
        return 'Neutro'
    elif v <= 0.6:
        return 'Positivo'
    else:
        return 'Molto_Positivo'

df14['valence_cat'] = df14['valence'].apply(lambda v: discretize_valence(v) if pd.notna(v) else None)

# Calcola matrice di transizione per ogni utente, poi media
transition_matrices = {}
categories = ['Molto_Negativo', 'Negativo', 'Neutro', 'Positivo', 'Molto_Positivo']

for sev in severities:
    users = df14[df14['severity'] == sev]['user_id'].unique()
    sev_transitions = np.zeros((5, 5))
    
    for uid in users:
        user_data = df14[(df14['user_id'] == uid) & df14['valence_cat'].notna()].sort_values('time')
        vals = user_data['valence_cat'].values
        
        for i in range(len(vals) - 1):
            from_idx = categories.index(vals[i])
            to_idx = categories.index(vals[i + 1])
            sev_transitions[from_idx, to_idx] += 1
    
    # Normalizza righe
    row_sums = sev_transitions.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    transition_matrices[sev] = (sev_transitions / row_sums).tolist()

print("\n═" * 60)
print("MATRICE DI TRANSIZIONE — ESEMPIO: SEVERA (20-27)")
print("═" * 60)
if 'Severa (20-27)' in transition_matrices:
    mat = np.array(transition_matrices['Severa (20-27)'])
    tm_df = pd.DataFrame(mat, index=categories, columns=categories)
    print(tm_df.round(3).to_string())
else:
    print("Nessun paziente con severità Severa nel dataset.")


═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
MATRICE DI TRANSIZIONE — ESEMPIO: SEVERA (20-27)
════════════════════════════════════════════════════════════
                Molto_Negativo  Negativo  Neutro  Positivo  Molto_Positivo
Molto_Negativo           0.587     0.276   0.117     0.016           0.005
Negativo                 0.233     0.405   0.303     0.053           0.006
Neutro                   0.054     0.229   0.587     0.126           0.005
Positivo                 0.039     0.143   0.418     0.369           0.031
Molto_Positivo           0.146     0.146   0.317     0.293           0.098


In [14]:
# ═══════════════════════════════════════════════════════════════
# 5.3  TASSO DI DROPOUT / MISSING DATA PER FASCIA
# ═══════════════════════════════════════════════════════════════
# Calcoliamo quante risposte mancano rispetto al teorico massimo (14 gg × 3 periodi = 42)

q_cols = [f'q{i}' for i in range(1, 15)] + ['q16', 'q46', 'q47']

for sev in severities:
    subset = df14[df14['severity'] == sev]
    if len(subset) > 0:
        total_possible = len(subset) * len(q_cols)
        total_missing = subset[q_cols].isna().sum().sum()
        missing_rate = total_missing / total_possible if total_possible > 0 else 0
        stats_by_severity[sev]['missing_rate'] = float(missing_rate)

print("\n═" * 60)
print("TASSO DI MISSING DATA PER FASCIA PHQ-9")
print("═" * 60)
for sev in severities:
    if sev in stats_by_severity and 'missing_rate' in stats_by_severity[sev]:
        mr = stats_by_severity[sev]['missing_rate']
        bar = '█' * int(mr * 50)
        print(f"  {sev:<22}  {mr*100:5.1f}%  {bar}")


═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
TASSO DI MISSING DATA PER FASCIA PHQ-9
════════════════════════════════════════════════════════════
  Minima (0-4)             87.3%  ███████████████████████████████████████████
  Lieve (5-9)              87.4%  ███████████████████████████████████████████
  Moderata (10-14)         87.3%  ███████████████████████████████████████████
  Mod. Severa (15-19)      87.2%  ███████████████████████████████████████████
  Severa (20-27)           87.3%  ███████████████████████████████████████████


In [15]:
# ═══════════════════════════════════════════════════════════════
# 5.4  DURATA MEDIA STREAK NEGATIVE
# ═══════════════════════════════════════════════════════════════
# Analizziamo per quanto tempo consecutivo un paziente rimane in zona negativa

streak_stats = {}
for sev in severities:
    users = df14[df14['severity'] == sev]['user_id'].unique()
    all_streaks = []
    
    for uid in users:
        user_data = df14[(df14['user_id'] == uid) & df14['valence'].notna()].sort_values('time')
        negative = (user_data['valence'] < 0).values
        
        current_streak = 0
        for is_neg in negative:
            if is_neg:
                current_streak += 1
            else:
                if current_streak > 0:
                    all_streaks.append(current_streak)
                current_streak = 0
        if current_streak > 0:
            all_streaks.append(current_streak)
    
    if all_streaks:
        streak_stats[sev] = {
            'mean_streak': float(np.mean(all_streaks)),
            'max_streak': int(np.max(all_streaks)),
            'median_streak': float(np.median(all_streaks)),
        }
        stats_by_severity[sev].update(streak_stats[sev])

print("\n═" * 60)
print("STREAK NEGATIVE CONSECUTIVE PER FASCIA")
print("═" * 60)
print(f"{'Severità':<22} {'Media':>8} {'Mediana':>8} {'Max':>6}")
print("─" * 50)
for sev in severities:
    if sev in streak_stats:
        s = streak_stats[sev]
        print(f"{sev:<22} {s['mean_streak']:>8.2f} {s['median_streak']:>8.1f} {s['max_streak']:>6}")


═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
STREAK NEGATIVE CONSECUTIVE PER FASCIA
════════════════════════════════════════════════════════════
Severità                  Media  Mediana    Max
──────────────────────────────────────────────────
Minima (0-4)               2.37      1.0     28
Lieve (5-9)                1.60      1.0     12
Moderata (10-14)           2.56      1.0     66
Mod. Severa (15-19)        3.11      2.0     98
Severa (20-27)             3.69      1.0    260


---
## 6. Salvataggio Artefatti per il Notebook 02

Salviamo tutti i pattern estratti in un unico file JSON che verrà consumato
dall'algoritmo di generazione sintetica nel prossimo notebook.

In [16]:
# ═══════════════════════════════════════════════════════════════
# 6.  SALVATAGGIO
# ═══════════════════════════════════════════════════════════════

extraction_artifact = {
    'metadata': {
        'created': datetime.now().isoformat(),
        'source_datasets': [
            'studentlife_phq9_scores.csv',
            'StudentLife_Mood (JSON)',
            '14day_ema_depression_log.csv'
        ],
        'description': 'Pattern statistici estratti dai dataset reali per la generazione sintetica'
    },
    'sintonia_valence_arousal_map': {k: {'valence': v, 'arousal': a} 
                                     for k, (v, a) in SINTONIA_VALENCE_AROUSAL.items()},
    'stats_by_severity': stats_by_severity,
    'transition_matrices': transition_matrices,
    'transition_categories': categories,
    'phq9_distribution': {
        'student_life_pre': {
            'mean': float(phq9_scored[phq9_scored['type']=='pre']['phq9_total'].mean()),
            'std': float(phq9_scored[phq9_scored['type']=='pre']['phq9_total'].std()),
            'median': float(phq9_scored[phq9_scored['type']=='pre']['phq9_total'].median()),
        },
        '14day': {
            'mean': float(patients_14d['phq9_total'].mean()),
            'std': float(patients_14d['phq9_total'].std()),
            'median': float(patients_14d['phq9_total'].median()),
        }
    }
}

artifact_path = os.path.join(RAW_DIR, 'statistical_patterns.json')
with open(artifact_path, 'w', encoding='utf-8') as f:
    json.dump(extraction_artifact, f, indent=2, ensure_ascii=False)

print(f"✅ Artefatto statistico salvato in: {artifact_path}")
print(f"   Dimensione: {os.path.getsize(artifact_path) / 1024:.1f} KB")
print(f"\n   Contenuto:")
print(f"   • Mappa Valence-Arousal SINTON-IA: {len(SINTONIA_VALENCE_AROUSAL)} stati")
print(f"   • Statistiche per {len(stats_by_severity)} fasce di severità")
print(f"   • Matrici di transizione: {len(transition_matrices)} fasce")
print(f"   • Distribuzione PHQ-9 da 2 dataset")

✅ Artefatto statistico salvato in: ..\data\raw\statistical_patterns.json
   Dimensione: 7.9 KB

   Contenuto:
   • Mappa Valence-Arousal SINTON-IA: 10 stati
   • Statistiche per 5 fasce di severità
   • Matrici di transizione: 5 fasce
   • Distribuzione PHQ-9 da 2 dataset


---
## 7. Riepilogo e Conclusioni

### Cosa abbiamo fatto in questo notebook:
1. **Definito la mappatura Valence-Arousal** per i 10 stati d'animo SINTON-IA
2. **Caricato e analizzato** il dataset Student Life (PHQ-9 + Mood JSON)
3. **Caricato e analizzato** il dataset 14-Day Depression Log
4. **Estratto pattern statistici** fondamentali:
   - Distribuzione valence per fascia di gravità
   - Volatilità emotiva
   - Matrici di transizione di Markov
   - Tassi di missing data (dropout)
   - Durata delle streak emotive negative

### Prossimo passo → Notebook 02
Tutti questi pattern verranno utilizzati per **generare 100.000 pazienti sintetici**
realistici, con stati d'animo mappati sulla scala 10×10 di SINTON-IA.